In [5]:
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd
import math
# Tải dữ liệu
from ucimlrepo import fetch_ucirepo
phishing_websites = fetch_ucirepo(id=327)
X = phishing_websites.data.features
y = phishing_websites.data.targets

# Chia tập train và test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
class Hyperband:
    def __init__(self, estimator, param_distributions, max_iter=81, eta=3, random_state=None):
        self.estimator = estimator
        self.param_distributions = param_distributions
        self.max_iter = max_iter  # Số lần lặp tối đa
        self.eta = eta  # Hệ số giảm
        self.random_state = random_state
        self.s_max = int(np.log(self.max_iter) / np.log(self.eta))
        self.B = (self.s_max + 1) * self.max_iter
        if self.random_state is not None:
            np.random.seed(self.random_state)
    
    def sample_params(self):
        """
        Lấy mẫu tham số từ các phân phối hoặc danh sách giá trị.
        Hỗ trợ:
            - scipy.stats distributions (randint, uniform,...)
            - list giá trị rời rạc
        """
        sampled_params = {}
        for param, dist in self.param_distributions.items():
            sampled_params[param] = dist.rvs()
        return sampled_params
    
    def successvivehalving(n, r, s, T, X, y)
    def fit(self, X, y):
        """Thực hiện tối ưu hóa siêu tham số với Hyperband"""
        best_score = -np.inf
        best_params = None
        results = []
        for s in reversed(range(self.s_max +1)):
            n = math.ceil(self.B / self.max_iter * self.eta ** s / (s + 1))
            r = self.max_iter * self.eta ** (-s)
            T = [self.sample_params() for _ in range(n)]     
            for i in range(s + 1):
                n_i = math.floor(n*n**(-i))
                r_i = int(r * self.eta**i)
                scores = []
                for t in T:
                    result = {}
                    if 'n_estimators' in t:
                        t['n_estimators'] = r_i
                    self.estimator.set_params(**t)
                    score = cross_val_score(self, estimator=self.estimator, X=X, y=y, cv=5, scoring='accuracy')
                    scores.append(score)
                    result['params'] = t
                    result['score'] = score
                k = math.floor(n_i / n)
                top_k_indices = np.argsort(scores)[-k:][::-1]
                T = T[top_k_indices]
        for result in results:
            if result['score'] > best_score:
                best_score = result['score']
                best_params = result['params']
        self.best_params_ = best_params
        self.best_score_ = best_score
        return self

In [7]:
from scipy.stats import randint, uniform

# Định nghĩa không gian tham số rộng hơn cho LightGBM
param_space = {
    'num_leaves': randint(20, 100),  # Số lá trong cây
    'max_depth': randint(3, 12),     # Độ sâu tối đa
    'learning_rate': uniform(0.01, 0.3),  # Tốc độ học
    'min_child_samples': randint(10, 50),  # Số mẫu tối thiểu trong mỗi lá
    'subsample': uniform(0.6, 0.4),   # Tỷ lệ mẫu sử dụng cho mỗi cây
    'colsample_bytree': uniform(0.6, 0.4),  # Tỷ lệ features sử dụng cho mỗi cây
    'reg_alpha': uniform(0, 1),       # L1 regularization
    'reg_lambda': uniform(0, 1),      # L2 regularization
    'min_child_weight': uniform(0, 1)  # Trọng số tối thiểu cho mỗi lá
}